# AI Lab 07: Computer Vision with OpenCV, MediaPipe, PyAutoGUI & TensorFlow

This comprehensive laboratory covers essential to advanced computer vision techniques:

1. **Part 1: Beginner OpenCV Practice** (Separate cells)
   - **Practice 1**: 2D Geometric Primitives (Lines, Circles, Rectangles, Ellipses, Text)
   - **Practice 2**: Color Spaces & Conversions (BGR -> RGB, BGR -> Grayscale, BGR -> HSV)
   - **Practice 3**: Canny Edge Detection & Sobel Gradients
   - **Practice 4**: Adding Noise (Gaussian & Salt-and-Pepper) and Noise Reduction (Gaussian, Median, Bilateral Filters)
   - **Practice 5**: Robust Camera Opening & Real-Time Face & Eye Detection using Haar Cascades

2. **Part 2: Task ii. Draw Realistic Pencil Sketch**
   - Grayscale -> Invert -> Gaussian Blur -> Color Dodge Division blend on static images & live camera.

3. **Part 3: Task i. Gesture Controls (MediaPipe + PyAutoGUI)**
   - Real-time hand landmark tracking and finger counting to control presentation slides, volume, and hotkeys.

4. **Part 4: Task iii. Virtual Air Canvas (MediaPipe + OpenCV)**
   - Interactive drawing in 3D air with color palette selection, eraser, and bitwise canvas blending.

5. **Part 5: Deep Learning & Vision with TensorFlow + OpenCV**
   - **Task 5A**: Real-time Live Object & Scene Classification on Webcam with MobileNetV2.
   - **Task 5B**: End-to-End Hand Gesture Classification CNN Pipeline with Keras & OpenCV.

> **Camera Controls**: In any live OpenCV video window, press `q` or `ESC` to safely close the camera.

## 0. Import Required Libraries and Verify Environment

In [ ]:
import os
import sys
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pyautogui
import mediapipe as mp
import tensorflow as tf

print("=== Environment Verification ===")
print(f"Python Version:     {sys.version.split()[0]}")
print(f"OpenCV Version:     {cv2.__version__}")
print(f"MediaPipe Version:  {mp.__version__}")
print(f"PyAutoGUI Version:  {pyautogui.__version__}")
print(f"TensorFlow Version: {tf.__version__}")
print("All required libraries are loaded and ready!")


## Part 1: Beginner OpenCV Practice

### Practice 1: Drawing Basics (Lines, Circles, Rectangles, Ellipse & Text)
Drawing 2D shapes on a NumPy array image using `cv2.line`, `cv2.circle`, `cv2.rectangle`, `cv2.ellipse`, and `cv2.putText`.

In [ ]:
# Create a blank black canvas (500x700 pixels, 3 color channels, uint8)
canvas = np.zeros((500, 700, 3), dtype=np.uint8)

# 1. Draw a diagonal Blue Line: cv2.line(img, pt1, pt2, color_bgr, thickness)
cv2.line(canvas, (50, 50), (350, 50), (255, 0, 0), thickness=4)

# 2. Draw a Green Rectangle: cv2.rectangle(img, pt1, pt2, color_bgr, thickness)
cv2.rectangle(canvas, (50, 90), (300, 220), (0, 255, 0), thickness=3)

# 3. Draw Circles: Red outline & Yellow filled (thickness=-1)
cv2.circle(canvas, (450, 150), 60, (0, 0, 255), thickness=3)           # Red outline
cv2.circle(canvas, (580, 150), 40, (0, 255, 255), thickness=-1)        # Yellow filled

# 4. Draw an Ellipse: cv2.ellipse(img, center, axes, angle, startAngle, endAngle, color, thickness)
cv2.ellipse(canvas, (180, 340), (100, 50), 30, 0, 360, (255, 0, 255), 2)

# 5. Add Text: cv2.putText(img, text, org, fontFace, fontScale, color, thickness)
cv2.putText(canvas, "OpenCV Drawing Basics", (50, 450), 
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)

# Display canvas using matplotlib (convert BGR to RGB)
plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("Practice 1: OpenCV Shape & Text Drawing")
plt.axis("off")
plt.show()


### Practice 2: Color Space Conversions (BGR to RGB & Grayscale)
OpenCV stores pixel data in **BGR** order, while Matplotlib and TensorFlow expect **RGB**. Grayscale drops the channels from 3 to 1.

In [ ]:
# Generate a colorful test synthetic image with colored bands
color_sample = np.zeros((300, 600, 3), dtype=np.uint8)
color_sample[:, 0:200] = [255, 0, 0]    # Blue in BGR
color_sample[:, 200:400] = [0, 255, 0]  # Green in BGR
color_sample[:, 400:600] = [0, 0, 255]  # Red in BGR

# 1. Convert BGR to RGB
img_rgb = cv2.cvtColor(color_sample, cv2.COLOR_BGR2RGB)

# 2. Convert BGR to Grayscale
img_gray = cv2.cvtColor(color_sample, cv2.COLOR_BGR2GRAY)

# 3. Convert BGR to HSV (Hue, Saturation, Value)
img_hsv = cv2.cvtColor(color_sample, cv2.COLOR_BGR2HSV)

# Display side-by-side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_rgb)
axes[0].set_title("BGR converted to RGB")
axes[0].axis("off")

axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title("Grayscale (Single Channel)")
axes[1].axis("off")

axes[2].imshow(img_hsv)
axes[2].set_title("HSV Color Space")
axes[2].axis("off")

plt.tight_layout()
plt.show()


### Practice 3: Canny Edge Detection & Sobel Gradients
Multi-stage Canny Edge Detector: Gaussian smoothing -> Gradient calculation -> Non-maximum suppression -> Hysteresis thresholding.

In [ ]:
# Create a test pattern image with geometric contours
test_img = np.zeros((300, 500), dtype=np.uint8)
cv2.rectangle(test_img, (50, 50), (200, 250), 255, -1)
cv2.circle(test_img, (350, 150), 80, 200, -1)
cv2.putText(test_img, "EDGES", (180, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 50, 3)

# Apply Gaussian Blur first to smooth high-frequency noise
blurred = cv2.GaussianBlur(test_img, (5, 5), 1.4)

# Apply Canny Edge Detector: cv2.Canny(image, threshold1, threshold2)
edges_canny = cv2.Canny(blurred, threshold1=50, threshold2=150)

# Sobel gradients for comparison
sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.magnitude(sobel_x, sobel_y)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(test_img, cmap='gray'); axes[0].set_title("Original Image"); axes[0].axis("off")
axes[1].imshow(edges_canny, cmap='gray'); axes[1].set_title("Canny Edge Detection"); axes[1].axis("off")
axes[2].imshow(sobel_combined, cmap='gray'); axes[2].set_title("Sobel Gradient Magnitude"); axes[2].axis("off")
plt.tight_layout()
plt.show()


### Practice 4: Noise Addition & Noise Reduction (Denoising Filters)
Demonstrating:
1. Adding synthetic Gaussian noise and Salt & Pepper noise.
2. Filtering with **Gaussian Blur**, **Median Blur** (best for impulse salt & pepper), and **Bilateral Filter** (edge-preserving).

In [ ]:
# Create clean image
base = np.zeros((300, 400), dtype=np.uint8)
for i in range(300):
    base[i, :] = int(i * 255 / 300)
cv2.circle(base, (200, 150), 70, 255, -1)
cv2.putText(base, "CLEAN", (130, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 0, 3)

# 1. Add Gaussian Noise
mean, sigma = 0, 25
gauss_noise = np.random.normal(mean, sigma, base.shape).astype(np.float32)
noisy_gauss = np.clip(base.astype(np.float32) + gauss_noise, 0, 255).astype(np.uint8)

# 2. Add Salt and Pepper Noise
noisy_sp = base.copy()
sp_ratio = 0.05
num_salt = int(sp_ratio * base.size * 0.5)
coords_salt = [np.random.randint(0, i - 1, num_salt) for i in base.shape]
noisy_sp[tuple(coords_salt)] = 255
coords_pepper = [np.random.randint(0, i - 1, num_salt) for i in base.shape]
noisy_sp[tuple(coords_pepper)] = 0

# 3. Noise Reduction Filters
denoised_gaussian = cv2.GaussianBlur(noisy_gauss, (5, 5), 1.5)
denoised_median = cv2.medianBlur(noisy_sp, 5)             # Highly effective for Salt & Pepper
denoised_bilateral = cv2.bilateralFilter(noisy_gauss, 9, 75, 75) # Preserves sharp edges

# Plot results
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].imshow(base, cmap='gray'); axes[0, 0].set_title("Original Clean"); axes[0, 0].axis("off")
axes[0, 1].imshow(noisy_gauss, cmap='gray'); axes[0, 1].set_title("Gaussian Noise Added"); axes[0, 1].axis("off")
axes[0, 2].imshow(noisy_sp, cmap='gray'); axes[0, 2].set_title("Salt & Pepper Noise Added"); axes[0, 2].axis("off")

axes[1, 0].imshow(denoised_gaussian, cmap='gray'); axes[1, 0].set_title("Gaussian Filtered"); axes[1, 0].axis("off")
axes[1, 1].imshow(denoised_median, cmap='gray'); axes[1, 1].set_title("Median Filtered (Clean S&P!)"); axes[1, 1].axis("off")
axes[1, 2].imshow(denoised_bilateral, cmap='gray'); axes[1, 2].set_title("Bilateral Filtered (Edges Kept)"); axes[1, 2].axis("off")
plt.tight_layout()
plt.show()


### Practice 5: Camera Opening & Robust Real-Time Face & Eye Detection (Haar Cascades)
This cell loads Haar Cascade classifiers for **Face** and **Eye** detection. It handles path resolution automatically across local directory and OpenCV data folders, with built-in auto-download if missing.

> **Controls**: Press `q` or `ESC` on the video window to quit.

In [ ]:
import os
import urllib.request
import cv2

def get_cascade(filename):
    """
    Searches for cascade file in local directory, cv2.data, or downloads it if absent.
    """
    candidates = [
        filename,
        os.path.join(getattr(cv2.data, 'haarcascades', ''), filename),
        os.path.join(os.getcwd(), filename)
    ]
    for p in candidates:
        if p and os.path.exists(p) and os.path.getsize(p) > 1000:
            cascade = cv2.CascadeClassifier(p)
            if not cascade.empty():
                print(f"Loaded cascade from: {p}")
                return cascade
                
    # Auto-download from official OpenCV repository
    url = f"https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/{filename}"
    print(f"Downloading {filename} from official repository...")
    urllib.request.urlretrieve(url, filename)
    cascade = cv2.CascadeClassifier(filename)
    return cascade

face_cascade = get_cascade('haarcascade_frontalface_default.xml')
eye_cascade = get_cascade('haarcascade_eye.xml')

if face_cascade.empty():
    print("Error: Failed to load face cascade.")
else:
    print("Face cascade is ready.")
    
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Warning: Could not open camera (camera index 0). Please ensure camera is connected and not used by another app.")
else:
    print("Camera opened successfully! Press 'q' in the OpenCV window to exit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame.")
            break
            
        # Mirror the frame horizontally
        frame = cv2.flip(frame, 1)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Detect faces
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(40, 40))
        
        for (x, y, w, h) in faces:
            # Draw Face rectangle (Green)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, "Face", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            # Detect eyes within the face ROI
            roi_gray = gray[y:y+h, x:x+w]
            roi_color = frame[y:y+h, x:x+w]
            if not eye_cascade.empty():
                eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=10, minSize=(15, 15))
                for (ex, ey, ew, eh) in eyes:
                    center = (ex + ew // 2, ey + eh // 2)
                    radius = int(round((ew + eh) * 0.25))
                    cv2.circle(roi_color, center, radius, (255, 0, 0), 2)  # Blue circle
        
        # Status overlay
        cv2.putText(frame, f"Faces Detected: {len(faces)} | Press 'q' to Quit", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)
        
        cv2.imshow("OpenCV Real-Time Face & Eye Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam closed successfully.")


---
## Part 2: Task ii. Draw Realistic Pencil Sketch Effect

### Pipeline for Pencil Sketch Conversion:
1. **Grayscale**: Convert the input image/frame to grayscale.
2. **Invert**: Invert the grayscale image (`255 - gray`).
3. **Gaussian Blur**: Blur the inverted image using a kernel (e.g. `25x25`).
4. **Color Dodge (Divide)**: Divide the grayscale image by the inverted blurred image (`cv2.divide(gray, 255 - blur, scale=256.0)`).

Runs live on the webcam displaying original feed and pencil sketch side-by-side.

In [ ]:
def convert_to_pencil_sketch(image, ksize=25):
    """
    Converts an input BGR image into an artistic pencil sketch.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    inverted = 255 - gray
    ksize = ksize if ksize % 2 == 1 else ksize + 1
    blurred = cv2.GaussianBlur(inverted, (ksize, ksize), sigmaX=0, sigmaY=0)
    sketch = cv2.divide(gray, 255 - blurred, scale=256.0)
    return sketch

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Camera not available for live pencil sketch demo.")
else:
    print("Starting Live Pencil Sketch. Press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.flip(frame, 1)
        sketch = convert_to_pencil_sketch(frame, ksize=25)
        sketch_bgr = cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)
        
        # Display original and sketch side-by-side
        combined = np.hstack((frame, sketch_bgr))
        cv2.putText(combined, "Original Webcam", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(combined, "Pencil Sketch Effect", (frame.shape[1] + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        
        cv2.imshow("Task ii: Pencil Sketch Effect", combined)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
    print("Pencil sketch demo finished.")


---
## Part 3: Task i. Gesture Controls (MediaPipe + PyAutoGUI)

In this task, we track hand landmarks and count raised fingers to trigger system automation shortcuts with `pyautogui`:
- **1 Finger (Index)**: Slide Next / Right Arrow (`pyautogui.press('right')`)
- **2 Fingers (Index + Middle)**: Slide Prev / Left Arrow (`pyautogui.press('left')`)
- **3 Fingers**: Volume Up (`pyautogui.press('volumeup')`)
- **4 Fingers**: Volume Down (`pyautogui.press('volumedown')`)
- **Fist (0 fingers)**: Mute / Idle (`pyautogui.press('volumemute')`)

Uses the local `hand_landmarker.task` model with automatic path lookup.

In [ ]:
import os
import time
import cv2
import mediapipe as mp
import pyautogui
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

pyautogui.FAILSAFE = True

# Resolve model path
model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    alt_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'hand_landmarker.task')
    if os.path.exists(alt_path):
        model_path = alt_path

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
landmarker = vision.HandLandmarker.create_from_options(options)
HAND_CONNECTIONS = vision.HandLandmarksConnections.HAND_CONNECTIONS

def count_fingers(landmarks):
    fingers = []
    # Thumb
    if landmarks[4].x < landmarks[3].x:
        fingers.append(1)
    else:
        fingers.append(0)
    # Index, Middle, Ring, Pinky
    for tip, pip in zip([8, 12, 16, 20], [6, 10, 14, 18]):
        if landmarks[tip].y < landmarks[pip].y:
            fingers.append(1)
        else:
            fingers.append(0)
    return fingers

def draw_hand_skeleton(frame, landmarks):
    h, w, _ = frame.shape
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]
    for conn in HAND_CONNECTIONS:
        if conn.start < len(pts) and conn.end < len(pts):
            cv2.line(frame, pts[conn.start], pts[conn.end], (0, 255, 0), 2)
    for pt in pts:
        cv2.circle(frame, pt, 4, (0, 0, 255), -1)

cap = cv2.VideoCapture(0)
last_action_time = 0
cooldown = 1.0
gesture_status = "Waiting for hand..."

print("Starting Gesture Control. Press 'q' to quit.")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    timestamp_ms = int(time.time() * 1000)
    results = landmarker.detect_for_video(mp_img, timestamp_ms)
    current_time = time.time()
    
    if results.hand_landmarks:
        for hand_lms in results.hand_landmarks:
            draw_hand_skeleton(frame, hand_lms)
            f = count_fingers(hand_lms)
            total_fingers = sum(f)
            
            if current_time - last_action_time > cooldown:
                if total_fingers == 1 and f[1] == 1:
                    pyautogui.press('right')
                    gesture_status = "ACTION: Next Slide (->)"
                    last_action_time = current_time
                elif total_fingers == 2 and f[1] == 1 and f[2] == 1:
                    pyautogui.press('left')
                    gesture_status = "ACTION: Previous Slide (<-"
                    last_action_time = current_time
                elif total_fingers == 3:
                    pyautogui.press('volumeup')
                    gesture_status = "ACTION: Volume Up (+)"
                    last_action_time = current_time
                elif total_fingers == 4:
                    pyautogui.press('volumedown')
                    gesture_status = "ACTION: Volume Down (-)"
                    last_action_time = current_time
                elif total_fingers == 0:
                    gesture_status = "Fist (Idle)"
            
            cv2.putText(frame, f"Fingers: {total_fingers}", (30, 80), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
    else:
        gesture_status = "No hand detected"
        
    cv2.putText(frame, gesture_status, (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    cv2.imshow("Task i: Gesture Control", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()
print("Gesture control terminated.")


---
## Part 4: Task iii. Virtual Air Canvas (MediaPipe + OpenCV)

### Features of the Virtual Canvas:
1. **Palette Bar**: Blue, Green, Red, Yellow, Eraser, and Clear All buttons.
2. **Selection Mode (Two fingers: Index + Middle)**: Pick color from palette without painting.
3. **Drawing Mode (One finger: Index only)**: Paint in 3D air.
4. **Overlay**: Blends strokes seamlessly over your camera feed.

In [ ]:
import os
import time
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = 'hand_landmarker.task'
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
landmarker = vision.HandLandmarker.create_from_options(options)

colors = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (0, 255, 255),  # Yellow
    (0, 0, 0)       # Eraser (Black)
]
color_names = ["BLUE", "GREEN", "RED", "YELLOW", "ERASER"]
current_color = colors[0]
brush_thickness = 8
eraser_thickness = 40

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

canvas = None
xp, yp = 0, 0

print("Starting Virtual Canvas. Press 'q' to quit.")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    frame = cv2.flip(frame, 1)
    h, w, c = frame.shape
    
    if canvas is None:
        canvas = np.zeros((h, w, 3), dtype=np.uint8)
        
    # Header UI
    btn_w = w // 6
    cv2.rectangle(frame, (0, 0), (w, 80), (50, 50, 50), -1)
    
    for i, name in enumerate(color_names):
        x1 = i * btn_w + 10
        x2 = (i + 1) * btn_w - 10
        col = colors[i] if i < 4 else (180, 180, 180)
        cv2.rectangle(frame, (x1, 10), (x2, 70), col, -1)
        text_col = (0, 0, 0) if i in [1, 3, 4] else (255, 255, 255)
        cv2.putText(frame, name, (x1 + 15, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_col, 2)
        
    cv2.rectangle(frame, (5 * btn_w + 10, 10), (w - 10, 70), (0, 0, 150), -1)
    cv2.putText(frame, "CLEAR", (5 * btn_w + 25, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    results = landmarker.detect_for_video(mp_img, int(time.time() * 1000))
    
    if results.hand_landmarks:
        lms = results.hand_landmarks[0]
        x_idx, y_idx = int(lms[8].x * w), int(lms[8].y * h)
        x_mid, y_mid = int(lms[12].x * w), int(lms[12].y * h)
        
        index_up = lms[8].y < lms[6].y
        middle_up = lms[12].y < lms[10].y
        
        # Selection Mode
        if index_up and middle_up:
            xp, yp = 0, 0
            cv2.circle(frame, (x_idx, y_idx), 15, (255, 255, 255), 2)
            cv2.putText(frame, "Selection Mode", (30, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            
            if y_idx < 80:
                btn_idx = x_idx // btn_w
                if btn_idx < 5:
                    current_color = colors[btn_idx]
                elif btn_idx == 5:
                    canvas = np.zeros((h, w, 3), dtype=np.uint8)
                    
        # Drawing Mode
        elif index_up and not middle_up:
            cv2.circle(frame, (x_idx, y_idx), 10, current_color, -1)
            cv2.putText(frame, "Drawing Mode", (30, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, current_color, 2)
            
            if xp == 0 and yp == 0:
                xp, yp = x_idx, y_idx
                
            thickness = eraser_thickness if current_color == (0, 0, 0) else brush_thickness
            cv2.line(canvas, (xp, yp), (x_idx, y_idx), current_color, thickness)
            xp, yp = x_idx, y_idx
        else:
            xp, yp = 0, 0
    else:
        xp, yp = 0, 0
        
    # Blend canvas
    gray_canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    _, inv_mask = cv2.threshold(gray_canvas, 10, 255, cv2.THRESH_BINARY_INV)
    frame_bg = cv2.bitwise_and(frame, frame, mask=inv_mask)
    output = cv2.bitwise_or(frame_bg, canvas)
    
    cv2.imshow("Task iii: Virtual Air Canvas", output)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()
print("Virtual Canvas session ended.")


---
## Part 5: TensorFlow & Deep Learning Vision with OpenCV

With **TensorFlow** installed, we complete the computer vision pipeline with two deep learning tasks:

1. **Task 5A: Live Real-Time Deep Learning Vision (MobileNetV2)**
   - Takes live webcam stream.
   - Highlights a target Region of Interest (ROI) box.
   - Preprocesses the ROI for TensorFlow (`cv2.resize -> BGR2RGB -> normalize -> batch expand`).
   - Performs real-time inference with MobileNetV2 and displays top predictions with confidence bars and live FPS.

2. **Task 5B: Custom Hand Gesture Recognition CNN (Keras + OpenCV)**
   - Defines a Convolutional Neural Network (CNN) architecture in Keras for classifying hand gestures (e.g. *Fist*, *Open Palm*, *Thumbs Up*, *Peace Sign*).
   - Demonstrates ROI extraction, preprocessing, model compiling, and inference pipeline.

### Task 5A: Real-Time Live Deep Learning Vision with TensorFlow (MobileNetV2 + OpenCV)
Place an object in the targeting box in the center of your webcam feed to classify it live in real-time.

In [ ]:
import time
import cv2
import numpy as np
import tensorflow as tf

print("Loading TensorFlow MobileNetV2 model...")
model = tf.keras.applications.MobileNetV2(weights='imagenet', input_shape=(224, 224, 3))
print("MobileNetV2 loaded successfully!")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Camera not accessible.")
else:
    print("Starting TensorFlow Live Vision. Press 'q' to quit.")
    fps_time = time.time()
    frame_count = 0
    fps = 0
    top_label, top_score = "Analyzing...", 0.0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        
        # Define Center ROI Box for classification
        box_size = 280
        x1 = max(0, (w - box_size) // 2)
        y1 = max(0, (h - box_size) // 2)
        x2 = min(w, x1 + box_size)
        y2 = min(h, y1 + box_size)
        roi = frame[y1:y2, x1:x2]
        
        # Preprocess ROI for TensorFlow every 3 frames for smooth FPS
        if frame_count % 3 == 0 and roi.size > 0:
            roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            roi_resized = cv2.resize(roi_rgb, (224, 224))
            tensor_in = tf.keras.applications.mobilenet_v2.preprocess_input(roi_resized.astype(np.float32))
            tensor_batch = np.expand_dims(tensor_in, axis=0)
            
            preds = model.predict(tensor_batch, verbose=0)
            decoded = tf.keras.applications.mobilenet_v2.decode_predictions(preds, top=1)[0][0]
            top_label = decoded[1].replace('_', ' ').title()
            top_score = decoded[2] * 100
            
        frame_count += 1
        if time.time() - fps_time >= 1.0:
            fps = frame_count / (time.time() - fps_time)
            fps_time = time.time()
            frame_count = 0
            
        # Draw Targeting Box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(frame, "Place Object in Box", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)
        
        # Draw Prediction Banner
        cv2.rectangle(frame, (20, 20), (450, 100), (0, 0, 0), -1)
        cv2.putText(frame, f"Prediction: {top_label}", (30, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
        cv2.putText(frame, f"Confidence: {top_score:.1f}% | FPS: {fps:.1f}", (30, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
        
        cv2.imshow("Task 5A: TensorFlow Live Inference with OpenCV", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
    print("Live TensorFlow vision session terminated.")


### Task 5B: Custom Hand Gesture Classification CNN (TensorFlow Keras + OpenCV)
Builds a specialized Convolutional Neural Network (CNN) architecture in Keras designed for classifying custom hand gestures directly from preprocessed OpenCV frames.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Gesture classes
GESTURE_CLASSES = ["Fist", "Open Palm", "Thumbs Up", "Peace Sign"]
NUM_CLASSES = len(GESTURE_CLASSES)
IMG_SIZE = (64, 64)

def build_gesture_cnn(input_shape=(64, 64, 3), num_classes=4):
    """
    Creates a lightweight CNN for real-time hand gesture recognition.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Dense Classifier Head
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

gesture_model = build_gesture_cnn(input_shape=(64, 64, 3), num_classes=NUM_CLASSES)
print("=== Custom Gesture CNN Architecture ===")
gesture_model.summary()

# Pipeline test with synthetic OpenCV frame
sample_hand_roi = np.random.randint(0, 256, (120, 120, 3), dtype=np.uint8)
roi_resized = cv2.resize(sample_hand_roi, IMG_SIZE)
roi_norm = roi_resized.astype(np.float32) / 255.0
batch_input = np.expand_dims(roi_norm, axis=0)

preds = gesture_model.predict(batch_input, verbose=0)[0]
pred_class_idx = np.argmax(preds)
confidence = preds[pred_class_idx] * 100

print(f"\nPipeline Verification:")
print(f"Input shape:      {batch_input.shape}")
print(f"Predicted Class:  {GESTURE_CLASSES[pred_class_idx]} ({confidence:.2f}% confidence)")
print(f"Classes supported: {GESTURE_CLASSES}")
